# 03 · Обучающая выборка

**Зачем этот шаг.** У нас есть входы (01: признаки и подграфы на каждую дату) и
ответы (02: оценки LLM по годам). Здесь мы соединяем их в таблицу, на которой
учится модель, и делаем это так, чтобы проверка качества была честной.

**Что такое пример.** Строка = технология на дату среза T:

- **входы** — признаки, посчитанные только по документам до T, плюс агрегаты
  соседей (как себя ведут технологии рядом);
- **ответ** `signal_llm` — 1, если LLM считает этот год слабым сигналом
  (оценка ≤ `weak_max`), 0 — если точно нет (≥ `not_weak_min`). Неуверенные годы и
  «не технологии» не размечаются.

Берутся только активные даты — где за год до T был хотя бы один документ.

Шаги ниже — те же, что в `src/lctrend/modeling/dataset/builder.py`, но по одному
и с проверкой каждого. Итоговые файлы пишет последняя ячейка тем же кодом, что
`scripts/build_dataset.py` и Docker-образ, — результат не зависит от способа запуска.

**Где мы в цепочке**

1. **01 · выгрузка** — что есть в графе на каждую дату (признаки и окружение технологий);
2. **02 · разметка** — что LLM считает слабым сигналом в каждый год;
3. **03 · выборка** — соединяем 01 и 02 в обучающую таблицу;
4. **04 · обучение** — CatBoost, HGT и их комбинация, метрики;
5. **05 · граф** — записываем итоговую разметку в узлы технологий.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"))
from lab import *
from IPython.display import display, Markdown

from datetime import date
from lctrend.modeling.dataset import builder
from lctrend.modeling.dataset.labels import FEATURES
from lctrend.modeling.dataset.llm_outcomes import llm_label_rows
from lctrend.modeling.dataset.neighbors import NEIGHBOR_FEATURES, iter_samples

TARGET = CONFIG["target"]
display(Markdown(
    f"Цель: `{TARGET}`. Пороги: слабый сигнал при оценке ≤ {CONFIG['labels']['weak_max']}, "
    f"не слабый при ≥ {CONFIG['labels']['not_weak_min']}; «не технологии» "
    f"{'исключаются' if CONFIG['labels']['drop_non_technologies'] else 'остаются'}."
))

Цель: `signal_llm`. Пороги: слабый сигнал при оценке ≤ 0.3, не слабый при ≥ 0.7; «не технологии» исключаются.

## 1. Соседи технологии

**Зачем.** Технология, вокруг которой всё растёт, ведёт себя иначе, чем
одиночка. CatBoost сам граф не видит, поэтому из подграфа каждой строки
считаем числа: сколько связанных технологий (родитель/потомок) и совместно
упомянутых, как быстро они растут, насколько они новые, сколько организаций
рядом и какая доля из них — компании.

**Как читать.** «Заполнено» — у скольких строк признак вообще посчитан (нет
соседей — пусто, а не ноль); «не ноль» — у скольких он содержательный.

In [2]:
history = builder.read_history(CONFIG["inputs"]["history"])
related = builder.enrich_and_relate(history, iter_samples(CONFIG["inputs"]["subgraphs"]))
neighbours = pd.DataFrame(history)[list(NEIGHBOR_FEATURES)].replace("", None).astype(float)
LABELS = {"nb_related_count": "связанных технологий (родитель/потомок)",
          "nb_comentioned_count": "совместно упомянутых технологий",
          "nb_growing_share": "доля растущих соседей",
          "nb_organization_count": "организаций рядом",
          "nb_company_share": "доля компаний среди организаций"}
view = pd.DataFrame({"заполнено": neighbours.notna().mean(), "не ноль": (neighbours.fillna(0) != 0).mean()}).loc[list(LABELS)]
display(view.rename(index=LABELS).style.format("{:.0%}"))

,заполнено,не ноль
связанных технологий (родитель/потомок),100%,4%
совместно упомянутых технологий,100%,77%
доля растущих соседей,77%,35%
организаций рядом,100%,89%
доля компаний среди организаций,89%,21%


## 2. Семейства: защита от подсказки через родственников

**Зачем.** Если «LoRA» попадёт в обучение, а «QLoRA» — в проверку, модель
узнает родственницу, и качество на проверке будет завышено. Поэтому
родственные технологии объединяем в семейства, и семейство целиком уходит
либо в обучение, либо в проверку.

**Что считается родством:** связь родитель–потомок в графе, а также пары из
журнала слияния — версии (GPT3-13B и GPT3-175B) и пары, которые судья счёл
разными, но очень похожими.

In [3]:
plan = json.loads(Path(CONFIG["inputs"]["duplicates_plan"]).read_text(encoding="utf-8"))
family_of = builder.families({row["technology_id"] for row in history}, [*related, *builder.plan_pairs(plan)])
first_seen = {}
for row in history:
    family = family_of[row["technology_id"]]
    seen = row.get("first_seen_date") or row["snapshot_date"]
    first_seen[family] = min(seen, first_seen.get(family, seen))
sizes = pd.Series(family_of).value_counts()
display(Markdown(
    f"**{num(len(sizes))}** семейств. Связей родитель–потомок: {num(len(related))}, пар из журнала слияния: "
    f"{num(len(builder.plan_pairs(plan)))}. Большинство семейств — одна технология; ниже — сколько семейств какого размера."
))
display(sizes.value_counts().sort_index().rename("семейств").rename_axis("технологий в семействе").head(10).to_frame())

**2 428** семейств. Связей родитель–потомок: 86, пар из журнала слияния: 3 079. Большинство семейств — одна технология; ниже — сколько семейств какого размера.

,семейств
технологий в семействе,
1,2243
2,125
3,23
4,14
5,9
6,3
7,1
8,2
12,1


## 3. Ответы: оценки LLM → классы

**Зачем смотрим.** Проверяем, сколько примеров каждого класса получилось и как
они распределены по годам: если все слабые сигналы окажутся в одном годе,
модель выучит год, а не сигнал.

**Как читать.** Строка — год среза, столбцы — сколько строк попало в каждую
категорию.

In [4]:
rule = {key: value for key, value in CONFIG["labels"].items() if key not in ("data_end", "llm_labels", "source", "note")}
labelled = llm_label_rows(history, builder.read_history(CONFIG["labels"]["llm_labels"]), rule)
frame = pd.DataFrame(labelled)
frame["год"] = frame.snapshot_date.str[:4]
BUCKET = {"weak_signal": "слабый сигнал (1)", "not_weak": "не слабый (0)",
          "unsure": "не уверен", "not_technology": "не технология"}
display(pd.crosstab(frame["год"], frame.llm_bucket.map(BUCKET)).tail(12))

llm_bucket,не слабый (0),не технология,не уверен,слабый сигнал (1)
год,,,,
2015,27,30,5,48
2016,47,39,13,58
2017,108,83,43,108
2018,166,175,105,159
2019,268,278,121,204
2020,464,614,258,312
2021,574,690,336,164
2022,717,709,337,197
2023,827,687,322,161


## 4. Разбиение на обучение, настройку и проверку

**Зачем.** Модель нельзя проверять на том, на чём она училась. Делим семейства
на три части:

- **train** — на них модель учится;
- **valid** — на них подбираются настройки (порог «сигнал/не сигнал»,
  калибровка вероятностей) и выбирается лучшая модель;
- **test** — трогаем только для итогового отчёта.

Семейства режутся **по дате первого появления**: в valid и test попадают только
технологии, которых модель не видела и которые появились позже обучающих.
Это имитирует реальную задачу — оценить *новую* технологию. Границы подбираются
так, чтобы доли семейств со слабым сигналом были близки к 70/15/15.

In [5]:
for row in labelled:
    row["family_id"] = family_of[row["technology_id"]]
rows = [row for row in labelled if row.get(TARGET) in ("0", "1")]
split = builder.assign_splits(rows, TARGET, CONFIG["split"]["fractions"], first_seen, CONFIG["split"]["embargo"])
display(Markdown(
    f"Граница valid: семейства, появившиеся с **{split['valid_start']}**; граница test: с **{split['test_start']}**."
))
counts = builder.class_counts(rows, TARGET)
parts = pd.DataFrame({part: {"строк «не слабый» (0)": v["rows"].get("0", 0), "строк «слабый сигнал» (1)": v["rows"].get("1", 0),
                             "семейств (0)": v["families"].get("0", 0), "семейств (1)": v["families"].get("1", 0)}
                      for part, v in counts.items() if part != "purged"}).T
display(parts.rename_axis("часть"))
display(Markdown(
    "**Как читать.** Главное — число *семейств* со слабым сигналом в valid и test: именно от него зависит, "
    "насколько надёжны будут метрики. Если их единицы, любая цифра качества — с очень широким разбросом."
))

Граница valid: семейства, появившиеся с **2021-11-25**; граница test: с **2023-07-12**.

,строк «не слабый» (0),строк «слабый сигнал» (1),семейств (0),семейств (1)
часть,,,,
train,3954,1346,415,248
valid,924,192,176,59
test,1192,144,291,44


**Как читать.** Главное — число *семейств* со слабым сигналом в valid и test: именно от него зависит, насколько надёжны будут метрики. Если их единицы, любая цифра качества — с очень широким разбросом.

## 5. Отбор признаков

**Зачем.** Из почти двух сотен признаков часть бесполезна: почти всегда пустые,
всегда одинаковые или почти копии друг друга. Они только запутывают модель.
Отбор делается **только по train**, чтобы информация из проверки не влияла на
то, какие признаки мы берём.

In [6]:
selection = builder.select_features(
    [row for row in rows if row["split"] == "train"], [*FEATURES, *NEIGHBOR_FEATURES],
    CONFIG["features"]["max_missing_share"], CONFIG["features"]["max_abs_correlation"])
reasons = pd.Series(selection["dropped"]).map(
    lambda r: "почти всегда пустой" if r == "missing" else "всегда одинаковый" if r == "constant" else "почти копия другого")
display(Markdown(f"Отобрано **{len(selection['selected'])}** признаков, отброшено {len(selection['dropped'])}:"))
display(reasons.value_counts().rename("признаков").rename_axis("причина").to_frame())

Отобрано **94** признаков, отброшено 12:

,признаков
причина,
всегда одинаковый,5
почти копия другого,4
почти всегда пустой,3


## 6. Сохранить выборку

Файлы пишет `build_dataset` — тот же код, что у скрипта и Docker-образа. Он
пересчитает все шаги выше заново и запишет в папку `dataset/` запуска:

- `training.csv` — размеченные строки с признаками, семейством и частью;
- `scoring.csv` — последняя дата каждой технологии, для итоговой разметки;
- `dataset.json` — что и из чего собрано (пороги, границы, признаки, числа классов).

Входные файлы (выгрузка и метки) не меняются; перезапуск просто пересоберёт выборку.

In [7]:
summary = builder.build_dataset(CONFIG)
display(Markdown(
    f"Готово: `{summary['files']['training']}`, признаков {len(summary['features'])}, "
    f"границы valid/test: {summary['split']['valid_start']} / {summary['split']['test_start']}."
))

Готово: `artifacts\modeling\2026-09-29-dedup\dataset\training.csv`, признаков 94, границы valid/test: 2021-11-25 / 2023-07-12.